In [ ]:

import torch
import math

# Reproducible example
torch.manual_seed(42)
print(torch.__version__)

# Configuration
seq_len = 5
d_model = 8

def sinusoidal_encoding(seq_len, d_model):
    # Position indices: [0, 1, ..., seq_len-1]
    positions = torch.arange(
        seq_len, dtype=torch.float32
    ).unsqueeze(1)

    # Even dimension indices: [0, 2, 4, ...]
    dimensions = torch.arange(
        0, d_model, 2, dtype=torch.float32
    )

    # Frequency for each sine/cosine pair
    div_term = torch.exp(
        -math.log(10000.0) * dimensions / d_model
    )

    angles = positions * div_term

    pe = torch.zeros(seq_len, d_model)

    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)

    return pe


# 1. Generate positional encodings
pe = sinusoidal_encoding(seq_len, d_model)

print("Positional encoding shape:", pe.shape)

# 2. Display the encoding matrix
print("\nPositional encoding:")
print(pe)

# 3. Verify position zero
expected = torch.tensor(
    [0., 1., 0., 1., 0., 1., 0., 1.]
)

assert torch.allclose(pe[0], expected, atol=1e-6)

print("\nPosition zero check: PASSED")

# 4. Different positions must have different vectors
assert not torch.allclose(pe[0], pe[1])

print("Position uniqueness check: PASSED")

# 5. Simulate Day 2 token embeddings
embedding = torch.nn.Embedding(
    num_embeddings=20,
    embedding_dim=d_model
)

token_ids = torch.tensor([[2, 5, 7, 3, 9],[1, 4, 6, 8, 0]])

print("\nToken IDs:", token_ids)

token_vectors = embedding(token_ids)

# Add positional information
transformer_input = token_vectors + pe.unsqueeze(0)

print("\nToken embedding shape:", token_vectors.shape)
print("Transformer input shape:", transformer_input.shape)


: 

In [ ]:

import torch

def apply_rope(x, positions, base=10000.0):
    """
    x:         [B, H, T, D], where D must be even
    positions: [T]
    """

    B, H, T, D = x.shape

    assert D % 2 == 0
    assert positions.numel() == T

    # Frequency for each pair of dimensions
    dim_indices = torch.arange(
        0, D, 2, device=x.device, dtype=x.dtype
    )

    inv_freq = 1.0 / (
        base ** (dim_indices / D)
    )

    # [T, D/2]
    angles = positions.to(x.dtype).unsqueeze(1) * inv_freq

    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]

    # Split into even and odd dimensions
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    # Rotate every pair
    rotated_even = x_even * cos - x_odd * sin
    rotated_odd = x_even * sin + x_odd * cos

    # Interleave even and odd dimensions
    output = torch.stack(
        [rotated_even, rotated_odd],
        dim=-1
    ).flatten(-2)

    return output


# Test configuration
B, H, T, D = 1, 2, 5, 8

x = torch.randn(B, H, T, D)

positions = torch.arange(T)

rotated = apply_rope(x, positions)

print("Input shape:", x.shape)
print("RoPE output shape:", rotated.shape)

# Check 1: Shape preservation
assert rotated.shape == x.shape

print("Shape check: PASSED")

# Check 2: Rotation preserves vector norms
original_norm = torch.linalg.vector_norm(x, dim=-1)
rotated_norm = torch.linalg.vector_norm(rotated, dim=-1)

print("Original norms:", original_norm)
print("Rotated norms:", rotated_norm)

assert torch.allclose(
    original_norm, rotated_norm, atol=1e-5
)

print("Norm preservation check: PASSED")

# Check 3: Position zero must not rotate the vector
assert torch.allclose(rotated[:, :, 0, :], x[:, :, 0, :])

print("Position zero check: PASSED")

# Check 4: A common shift of both positions
# should preserve the RoPE query-key dot product.

q = torch.randn(1, 1, 1, D)
k = torch.randn(1, 1, 1, D)

q_pos2 = apply_rope(q, torch.tensor([2]))
k_pos4 = apply_rope(k, torch.tensor([4]))

q_pos5 = apply_rope(q, torch.tensor([5]))
k_pos7 = apply_rope(k, torch.tensor([7]))

score1 = (q_pos2 * k_pos4).sum()
score2 = (q_pos5 * k_pos7).sum()

assert torch.allclose(score1, score2, atol=1e-5)

print("Relative position check: PASSED")
